# 10 — EcoCAR SUPER PIPELINE: Joint Detection & Lane Segmentation

This standalone "Super Notebook" consolidates Data Parsing, Mask Generation, Model Training, and Inference into a single uninterrupted sequence.

By keeping everything in ONE browser tab, we **completely eliminate** Google Drive temporary storage disconnects and cross-VM wipeouts.

## 1. Core Setup & Environment

In [17]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib pyyaml tqdm

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    print(f"\n✅ GPU Accelerated: {torch.cuda.get_device_name(0)}")
else:
    print("\n❌ No GPU detected! Training will be impossible.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ GPU Accelerated: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Load Neural Network Code (GitHub)

In [18]:
import sys, os

# Safely clone the GitHub repository for model utilities
!rm -rf /content/ecocar
!git clone https://github.com/ChenSiyun1234/EcoCAR-Perception-Pipeline-YOLO26-BDD100K.git /content/ecocar

PROJECT_SRC = "/content/ecocar/src"
if os.path.isdir(PROJECT_SRC):
    sys.path.insert(0, os.path.dirname(PROJECT_SRC))
    print("✅ Added Neural Network Source Code to Python Path")

Cloning into '/content/ecocar'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 60 (delta 22), reused 49 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 91.25 KiB | 3.65 MiB/s, done.
Resolving deltas: 100% (22/22), done.
✅ Added Neural Network Source Code to Python Path


## 3. High-Speed Local Dataset Assembly (Replaces Nb 01, 02, 07)

This block aggressively searches the raw BDD100K ZIP extracts on Google Drive, and writes all required `images/`, `labels/`, and `masks/` to the local Colab NVMe SSD.

> **Crucially:** It is strictly guarded against FUSE network timeouts by utilizing pinpoint addressing and skipping mass-directory scans.

In [19]:
!find /content/drive/MyDrive/EcoCAR -name "0bc4a672-7ab359c4.jpg" 2>&1


/content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw/100k/train/0bc4a672-7ab359c4.jpg


In [20]:
import os, json, numpy as np, cv2, shutil
from pathlib import Path
from tqdm.auto import tqdm

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
RAW = f"{ECOCAR_ROOT}/datasets/bdd100k_raw"
YOLO = "/content/bdd100k_yolo"

# Reset NVMe space safely
shutil.rmtree(YOLO, ignore_errors=True)
for split in ['train', 'val']:
    for f in ['images', 'labels', 'masks']:
        os.makedirs(f"{YOLO}/{f}/{split}", exist_ok=True)

# ── Intelligent JSON Locator ──
def find_master_json(base_dir):
    for root, dirs, files in os.walk(base_dir):
        if '100k' in dirs: dirs.remove('100k')
        if 'images' in dirs: dirs.remove('images')
        for f in files:
            if 'train' in f and f.endswith('.json') and 'val' not in f and 'lane' not in f and 'det' not in f:
                return os.path.join(root, f)
    return None

json_path = find_master_json(RAW)
if not json_path:
    raise FileNotFoundError("❌ Fatal: The BDD100K Train Master JSON could not be traced. Ensure the release ZIP is unzipped.")

print(f"✅ Master Blueprint found: {json_path}")

with open(json_path, 'r') as f:
    data = json.load(f)
    sample_name = data[0].get('name', '') if len(data) > 0 else None

# ── Intelligent Image Linker ──
if sample_name:
    if not sample_name.endswith('.jpg'): sample_name += '.jpg'
    candidate_dirs = [
        f"{RAW}/images/100k/train",
        f"{RAW}/100k/train",
        f"{RAW}/bdd100k/images/100k/train"
    ]
    true_img_path = next((d for d in candidate_dirs if os.path.exists(os.path.join(d, sample_name))), None)

    if not true_img_path:
        raise FileNotFoundError(f"❌ Fatal: Tested real image '{sample_name}', but it doesn't physically exist in any standard Drive paths.")
    print(f"✅ True Image Vault locked: {true_img_path}")

    LOCAL_IMG = "/content/bdd100k_images/100k/train"
    os.makedirs(os.path.dirname(LOCAL_IMG), exist_ok=True)
    os.symlink(true_img_path, LOCAL_IMG)

# ── Ultra-Fast Dataset Renderer ──
BDD_LANE_CATEGORIES = ["lane/crosswalk", "lane/double other", "lane/double white", "lane/double yellow", "lane/road curb", "lane/single other", "lane/single white", "lane/single yellow"]
LANE_CAT = {cat: 1 for cat in BDD_LANE_CATEGORIES}
CLASS_TO_ID = {name: idx for idx, name in enumerate(["person", "rider", "car", "truck", "bus", "train", "motorcycle", "bicycle", "traffic light", "traffic sign"])}

def _bezier_curve(p0, p1, p2, p3):
    pts = []
    for i in range(31):
        t = i / 30; mt = 1 - t
        x = mt**3*p0[0] + 3*mt**2*t*p1[0] + 3*mt*t**2*p2[0] + t**3*p3[0]
        y = mt**3*p0[1] + 3*mt**2*t*p1[1] + 3*mt*t**2*p2[1] + t**3*p3[1]
        pts.append((int(round(x)), int(round(y))))
    return pts

def _poly2d_to_points(vertices, types):
    points = []; i = 0; n = len(vertices)
    while i < n:
        vx = vertices[i][0] * 320/1280; vy = vertices[i][1] * 180/720
        if i + 2 < n and i + 1 < len(types) and types[i + 1] == 'C' and i + 3 < n:
            p0 = (vx, vy)
            p1 = (vertices[i+1][0]*320/1280, vertices[i+1][1]*180/720)
            p2 = (vertices[i+2][0]*320/1280, vertices[i+2][1]*180/720)
            p3 = (vertices[i+3][0]*320/1280, vertices[i+3][1]*180/720)
            points.extend(_bezier_curve(p0, p1, p2, p3))
            i += 4
        else:
            points.append((int(round(vx)), int(round(vy))))
            i += 1
    return points

for frame in tqdm(data, desc="Building Single-Shot BDD Dataset (Local NVMe)"):
    name = frame.get('name', '')
    if not name.endswith('.jpg'): name += '.jpg'
    stem = Path(name).stem
    labels = frame.get('labels') or []

    lines = []
    has_lane = False
    for lb in labels:
        cat = lb.get('category', '')
        if cat in CLASS_TO_ID and 'box2d' in lb:
            box = lb['box2d']
            xc = ((box['x1']+box['x2'])/2) / 1280; yc = ((box['y1']+box['y2'])/2) / 720
            w = (box['x2']-box['x1']) / 1280; h = (box['y2']-box['y1']) / 720
            lines.append(f"{CLASS_TO_ID[cat]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        if cat in LANE_CAT: has_lane = True

    if not lines and not has_lane: continue

    # Generate YOLO BBox file
    with open(f"{YOLO}/labels/train/{stem}.txt", 'w') as f: f.write('\n'.join(lines))

    # Generate exact JPG symlink (FUSE-proof format)
    src_img = f"{LOCAL_IMG}/{name}"
    dst_img = f"{YOLO}/images/train/{name}"
    os.symlink(src_img, dst_img)

    # Rasterize Segmentation Lane Mask
    mask = np.zeros((180, 320), dtype=np.uint8)
    if has_lane:
        for lb in labels:
            if lb.get('category', '') in LANE_CAT:
                poly2d_data = lb.get('poly2d')
                if not poly2d_data or not isinstance(poly2d_data, list): continue
                polygons = []
                if isinstance(poly2d_data[0], dict): polygons = poly2d_data
                elif isinstance(poly2d_data[0], list) and len(poly2d_data[0]) >= 2 and isinstance(poly2d_data[0][0], (int, float)): polygons = [{'vertices': poly2d_data}]
                elif isinstance(poly2d_data[0], list) and len(poly2d_data[0]) > 0 and isinstance(poly2d_data[0][0], list): polygons = [{'vertices': p} for p in poly2d_data if isinstance(p, list)]
                for poly in polygons:
                    verts = poly.get('vertices', [])
                    if len(verts) < 2: continue
                    pts = _poly2d_to_points(verts, poly.get('types', 'L'*len(verts)))
                    if len(pts) >= 2: cv2.polylines(mask, [np.array(pts, dtype=np.int32)], False, 255, 2)
    cv2.imwrite(f"{YOLO}/masks/train/{stem}.png", mask)

print("✅ Unified File Assembly Complete! Everything compiled purely locally.")

✅ Master Blueprint found: /content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw/labels/bdd100k_labels_images_train.json
✅ True Image Vault locked: /content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw/100k/train


FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw/100k/train' -> '/content/bdd100k_images/100k/train'

## 4. Construct Neural Network Architecture

In [21]:
from src.multitask_model import build_multitask_model

model = build_multitask_model(
    weights='yolo26s.pt',  # Using specialized lightweight scalable YOLO26 backbone
    mask_height=180,
    mask_width=320,
    lane_hidden_channels=64,
)

print(f"✅ Model instantiated dynamically with parameter count: {sum(p.numel() for p in model.parameters()):,}")

✅ MultiTaskYOLO built:
   Backbone+Neck layers: 23
   Detect head: Detect
   Neck output indices: [16, 19, 22]
   Neck channels: [128, 256, 512]
   Lane head params: 571,745
✅ Model instantiated dynamically with parameter count: 10,581,529


## 5. DataLoader & Safety Sanity Check
> **Notice:** We are explicitly setting `num_workers=0`.
Even though your hardware implies multiple threads (e.g. `num_workers=4`), doing so with images hosted across the Google Drive FUSE bridge forces the connection to snap violently and return a FileNotFoundError loop. By relying on a linear thread, we enforce a slow and steady data load completely safeguarding the training lifecycle.

In [22]:
from src.joint_dataset import JointBDDDataset, joint_collate_fn
from torch.utils.data import DataLoader

train_dataset = JointBDDDataset(
    images_dir=f"{YOLO}/images/train",
    labels_dir=f"{YOLO}/labels/train",
    masks_dir=f"{YOLO}/masks/train",
    img_size=640,
    mask_height=180,
    mask_width=320,
    augment=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,  # 🔥 CRITICAL: Must be exactly 0 to stop Google Drive timeouts over PyTorch IPC!!
    collate_fn=joint_collate_fn,
    pin_memory=True,
    drop_last=True,
)

print(f"✅ Core Dataloaders primed. Sample Yield: {len(train_dataset)} Total Matrices / {len(train_loader)} Batches")

✅ JointBDDDataset: 0 samples
   Images: /content/bdd100k_yolo/images/train
   Labels: /content/bdd100k_yolo/labels/train
   Masks:  /content/bdd100k_yolo/masks/train


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
import matplotlib.pyplot as plt

print("Drawing single verification batch...")
batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i in range(min(4, batch['images'].shape[0])):
    img = batch['images'][i].permute(1, 2, 0).numpy()
    mask = batch['lane_masks'][i, 0].numpy()
    axes[0, i].imshow(img); axes[0, i].axis('off'); axes[0, i].set_title(f"Img {i}")
    axes[1, i].imshow(mask, cmap='gray'); axes[1, i].axis('off'); axes[1, i].set_title(f"Lane Seg {i}")

plt.suptitle('Training Batch Verification Diagnostics', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Joint Training Loop

Commence multi-task parameter convergence optimizing across BBox and Mask domains natively.

In [ ]:
from src.joint_trainer import JointTrainer, plot_joint_training_history

device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = JointTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=train_loader, # Temporary fallback since we skipped Val generation for velocity
    det_loss_weight=1.0,
    lane_loss_weight=0.5,
    lr=1e-3,
    weight_decay=1e-4,
    device=device,
    amp=True,
    save_dir=f"{ECOCAR_ROOT}/training_runs/joint_det_lane",
    save_period=5,
)

print("🔥 Starting Engine: Commencing 10 Epochs of Joint Training!")
history = trainer.train(epochs=10)

plot_joint_training_history(history, save_path=f"{ECOCAR_ROOT}/training_runs/joint_det_lane/history.png")
print("✅ Training cycle complete and histories safely logged in Drive.")

## 7. Model Inference Assessment
Evaluate inference capability through direct load validations against final `.pt` weights arrays.

In [ ]:
from src.multitask_model import MultiTaskYOLO
import torchvision.transforms as T

save_dir = f"{ECOCAR_ROOT}/training_runs/joint_det_lane"
best_weights = os.path.join(save_dir, 'weights', 'best.pt')

if os.path.exists(best_weights):
    eval_model = build_multitask_model(weights='yolo26s.pt', mask_height=180, mask_width=320, lane_hidden_channels=64)
    eval_model.load_state_dict(torch.load(best_weights, map_location=device)['model_state'])
    eval_model.to(device).eval()

    # Inference verification run against random index
    test_img = batch['images'][0].to(device).unsqueeze(0)
    with torch.no_grad():
        preds = eval_model(test_img)

    # Parse logits
    lane_mask_logits = preds['lane_masks']
    pred_mask = torch.sigmoid(lane_mask_logits[0, 0]).cpu().numpy() > 0.5

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.imshow(test_img[0].cpu().permute(1, 2, 0).numpy())
    plt.title('Baseline RGB'); plt.axis('off')

    plt.subplot(1, 2, 2); plt.imshow(pred_mask, cmap='magma')
    plt.title('NN Inferred Autonomous Mask'); plt.axis('off')
    plt.show()
else:
    print(f"⚠️ Weights not found at {best_weights}. Has training physically finished processing?")